# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


## My lane as an ML task (type)

**Task type: Ranking / Scoring**, with a binary proxy used underneath for evaluation.

The output my lane needs is a ranked queue — "review this page before that one" — not a single
yes/no label in isolation. That points to ranking/scoring rather than plain classification.
But to train and evaluate a score, it helps to have a binary proxy target under the hood (did
this page get flagged as worth reviewing, or not) so I can compute precision@K, the way Lane 2's
guide recommends. So: **scoring produces the output, a binary proxy makes it trainable and
checkable.** It is not clustering (I have a specific priority in mind, not an open-ended
grouping) and it is not pure classification (the deliverable is an ordering, not a single
per-page verdict).


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or proxy

**Proxy target: `at_risk`** — a page that (a) had real, measurable demand a month ago, (b) lost a
large share of that demand in the most recent 30 days, and (c) still holds a defensible ranking
position, so there's something worth protecting.

I deliberately do **not** use the precomputed `trend_direction` / `trend_pct` columns for this —
those are derived fields, and the flyrank data skill flags them as off-limits for feature-building
because a label made from them just teaches a model to recover a rule that's already baked into
the data (a circular result). Instead I build the proxy myself from two raw, independently
measured windows: `impressions_prev_30d` and `impressions_last_30d`. That keeps the label an
**observed outcome** (what actually happened to real numbers over two real time windows), not a
rule someone already computed for me.

Working definition: `impressions_prev_30d > 20` (there was real prior demand) **and**
`avg_position` between 1 and 20 (still ranked, so worth defending) **and** the drop from
prior-30d to last-30d impressions is 40% or worse. I show this built as a real column in Section 4.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

# Success metric
**Precision@K** (K = top 20 or top 50 of the ranked queue), with recall and average precision as
supporting numbers.

I'm picking precision@K over plain accuracy because the real workflow only ever looks at the top
of the queue — an editor doesn't work through all 30,000 rows, they work through the top 20 or 50.
A metric that credits the model for being right about the bottom of the list (which nobody will
ever read) doesn't measure anything useful here. Precision@K asks the right question directly:
"of the pages I told the editor to look at first, how many were actually worth it?" I'll also
track average precision across the full ranking to make sure the ordering is sensible beyond just
the very top, and I'll do a by-hand read of the top 20 in a later notebook, since a metric alone
can hide obviously wrong picks.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The cell below loads the starter slice, keeps only the columns relevant to this lane, and builds
the `at_risk` proxy column described above so you can see it as real data rather than an abstract
definition.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import os
import shutil

local_path = "../../data/raw/content_refresh_anonymized.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    # Remove any stale/incomplete clone first
    if os.path.exists("internship"):
        shutil.rmtree("internship")
    !git clone https://github.com/UnzilaAhsan/internship.git
    df = pd.read_csv("internship/data/raw/content_refresh_anonymized.csv")

# Keep only lane-relevant columns: identity, demand, position, freshness.
# Note: trend_direction / trend_pct are intentionally excluded — they're derived
# fields that would make any label built from them circular.
lane_cols = [
    "content_id", "client_id", "content_type",
    "impressions_prev_30d", "impressions_last_30d",
    "avg_position", "position_tier", "freshness_tier", "days_since_last_update",
]
lane_df = df[lane_cols].copy()

# Build the proxy target from raw, independently-measured windows.
lane_df["impr_change_pct"] = (
    (lane_df["impressions_last_30d"] - lane_df["impressions_prev_30d"])
    / lane_df["impressions_prev_30d"].replace(0, pd.NA) * 100
)

lane_df["at_risk"] = (
    (lane_df["impressions_prev_30d"] > 20)
    & (lane_df["avg_position"] > 0)
    & (lane_df["avg_position"] <= 20)
    & (lane_df["impr_change_pct"] <= -40)
).astype(int)

print(f"Rows: {len(lane_df):,}  (one row = one content item)")
print(f"at_risk base rate: {lane_df['at_risk'].mean()*100:.1f}% "
      f"({lane_df['at_risk'].sum():,} of {len(lane_df):,} pages)")
lane_df.head(10)



Cloning into 'internship'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 141 (delta 50), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.86 MiB | 16.26 MiB/s, done.
Resolving deltas: 100% (50/50), done.
Rows: 30,000  (one row = one content item)
at_risk base rate: 22.4% (6,728 of 30,000 pages)


,content_id,client_id,content_type,impressions_prev_30d,impressions_last_30d,avg_position,position_tier,freshness_tier,days_since_last_update,impr_change_pct,at_risk
0,content_304f48230142,client_f369cb89fc,keyword article,987,578,10.6,striking,0-30,20,-41.438703,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,5915,2501,20.3,page_3_5,0-30,25,-57.717667,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,6089,2382,36.5,page_3_5,0-30,20,-60.880276,0
3,content_331d6c4de07b,client_19581e27de,keyword article,4206,3626,6.2,page_1,0-30,22,-13.789824,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,6452,4211,44.0,page_3_5,0-30,14,-34.733416,0
5,content_d4084a4bc775,client_f369cb89fc,keyword article,1009,617,8.5,page_1,0-30,20,-38.850347,0
6,content_9a34b442b552,client_8722616204,keyword article,13,1,7.0,page_1,0-30,20,-92.307692,0
7,content_a63219c6e95a,client_19581e27de,keyword article,632,636,21.2,page_3_5,0-30,22,0.632911,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,13828,5696,46.0,page_3_5,0-30,20,-58.808215,0
9,content_c27558df2b0c,client_19581e27de,keyword article,356,252,4.9,page_1,91-180,104,-29.213483,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule (e.g. "flag anything with a 40%+ drop") is exactly what I used above to *build the
label* — and that's the point: it's a reasonable starting definition, but it's blunt. It treats
every 40%+ drop identically, whether it's on a page 3 slots from page one or buried at position
18; whether it's a one-off content type with naturally noisy traffic or a stable evergreen page;
whether it's freshly published (some volatility expected) or years old (a real signal). These
factors interact rather than stack neatly, which is exactly the case where a single if-statement
either over-flags noisy-but-fine pages or under-flags real problems sitting just under the
threshold. A model (even a simple one, like the baseline score and logistic regression Lane 2
calls for) can weigh several of these signals *together* and produce a continuous, rankable score
instead of a brittle yes/no cutoff — while still staying explainable enough to hand an editor a
reason code, not just a number.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.